## Config

In [1]:
import os
import csv
import math
import random
import json
import glob
from datetime import datetime
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from torch.amp import GradScaler, autocast

DATA_DIR = 'data'
VOCAB_PATH = os.path.join(DATA_DIR, 'vocab.txt')
META_CSV = os.path.join(DATA_DIR, 'metadata_clean.csv')
TOK_DIR = os.path.join(DATA_DIR, 'tokenized')

# ---------- Training config ----------
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEQ_LEN = 50        # input length
STRIDE = 10         # step between windows (reduce redundancy)
BATCH_SIZE = 256
EPOCHS = 8
LR = 1e-3
GRAD_CLIP = 1.0

EMB_DIM = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 2
DROPOUT = 0.2

# Per-book split ratios (each book is split into 3 contiguous parts)
TRAIN_RATIO = 0.8
VAL_RATIO   = 0.1
TEST_RATIO  = 0.1

MIN_BOOK_TOKENS = 2000  # skip tiny books

STEPS_PER_EPOCH = 3000

# ---------- Checkpoint config ----------
CKPT_DIR = 'checkpoints'
SAVE_EVERY_STEPS = 100          # save a step checkpoint every N steps (crash recovery)
PRUNE_VAL_LOSS_THRESHOLD = 0.1  # after epoch, delete epoch ckpts with val_loss > best + this

# Resume: set to a checkpoint .pt path to resume training, or None for fresh start
RESUME_FROM = None  # e.g. 'checkpoints/20260311143022/epoch_003.pt'
# ------------------------------------

use_amp = (DEVICE == "cuda")

print('DEVICE:', DEVICE)
os.makedirs(CKPT_DIR, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE: cuda


## Load vocab + metadata

In [2]:
def load_vocab(vocab_path: str) -> Tuple[List[str], Dict[str, int]]:
    if not os.path.exists(vocab_path):
        raise FileNotFoundError(f'vocab.txt not found: {vocab_path}')
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f if line.strip()]
    tok2id = {t: i for i, t in enumerate(vocab)}
    return vocab, tok2id

def read_metadata(meta_csv: str) -> List[dict]:
    if not os.path.exists(meta_csv):
        raise FileNotFoundError(f'metadata.csv not found: {meta_csv}')
    with open(meta_csv, 'r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))

vocab, tok2id = load_vocab(VOCAB_PATH)
rows = read_metadata(META_CSV)

pad_id = tok2id.get('<pad>', 0)
bos_id = tok2id.get('<bos>')
eos_id = tok2id.get('<eos>')

print('vocab size:', len(vocab))
print('special ids:', {'<pad>': pad_id, '<bos>': bos_id, '<eos>': eos_id})
print('metadata rows:', len(rows))

vocab size: 26286
special ids: {'<pad>': 0, '<bos>': 2, '<eos>': 3}
metadata rows: 49


## Dataset (windowed next-token prediction)

In [3]:
def load_ids_file(path: str) -> List[int]:
    with open(path, "r", encoding="utf-8") as f:
        s = f.read().strip()
    return list(map(int, s.split())) if s else []

@dataclass
class BookData:
    ebook_id: str
    ids_path: str
    token_count: int

def collect_books(rows: List[dict]) -> List[BookData]:
    books = []
    for r in rows:
        ebook_id = r.get('ebook_id')
        if not ebook_id:
            continue
        ids_path = os.path.join(TOK_DIR, f'{ebook_id}.ids.txt')
        if not os.path.exists(ids_path):
            continue
        try:
            token_count = int(r.get('token_count', '0'))
        except Exception:
            token_count = 0
        if token_count < MIN_BOOK_TOKENS:
            continue
        books.append(BookData(str(ebook_id), ids_path, token_count))
    return books

def split_book_tokens(ids: List[int], train_r: float, val_r: float) -> Tuple[List[int], List[int], List[int]]:
    """Split a single book's tokens into train/val/test contiguous segments."""
    n = len(ids)
    train_end = int(n * train_r)
    val_end = int(n * (train_r + val_r))
    return ids[:train_end], ids[train_end:val_end], ids[val_end:]

class CachedWindowDataset(Dataset):
    def __init__(self, token_chunks: List[torch.Tensor], seq_len: int, stride: int):
        self.seq_len = seq_len
        self.stride = stride
        self.book_tokens = []
        self.samples: List[Tuple[int, int]] = []

        for chunk in token_chunks:
            if chunk.numel() < seq_len + 2:
                continue
            bi = len(self.book_tokens)
            self.book_tokens.append(chunk)
            max_start = chunk.numel() - (seq_len + 1)
            for st in range(0, max_start + 1, stride):
                self.samples.append((bi, st))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        bi, st = self.samples[idx]
        ids = self.book_tokens[bi]
        chunk = ids[st: st + self.seq_len + 1].to(torch.long)
        return chunk[:-1], chunk[1:]

# --- Collect books and split each book's tokens ---
books = collect_books(rows)
if not books:
    raise RuntimeError('No tokenized books found. Check data/tokenized/*.ids.txt and metadata.csv token_count.')

train_chunks = []
val_chunks = []
test_chunks = []

total_train_tokens = 0
total_val_tokens = 0
total_test_tokens = 0

for b in books:
    ids = load_ids_file(b.ids_path)
    tr, va, te = split_book_tokens(ids, TRAIN_RATIO, VAL_RATIO)
    train_chunks.append(torch.tensor(tr, dtype=torch.int32))
    val_chunks.append(torch.tensor(va, dtype=torch.int32))
    test_chunks.append(torch.tensor(te, dtype=torch.int32))
    total_train_tokens += len(tr)
    total_val_tokens += len(va)
    total_test_tokens += len(te)

train_ds = CachedWindowDataset(train_chunks, SEQ_LEN, STRIDE)
val_ds   = CachedWindowDataset(val_chunks, SEQ_LEN, STRIDE)
test_ds  = CachedWindowDataset(test_chunks, SEQ_LEN, STRIDE)

print(f'Books: {len(books)} (each split {TRAIN_RATIO:.0%}/{VAL_RATIO:.0%}/{TEST_RATIO:.0%})')
print(f'Tokens: train={total_train_tokens:,}  val={total_val_tokens:,}  test={total_test_tokens:,}')
print(f'Samples: train={len(train_ds):,}  val={len(val_ds):,}  test={len(test_ds):,}')

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Books: 49 (each split 80%/10%/10%)
Tokens: train=2,592,401  val=324,048  test=324,076
Samples: train=259,019  val=32,185  test=32,187


## Model

In [4]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, num_layers: int, dropout: float, pad_id: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)          # (B,T,E)
        out, _ = self.lstm(e)    # (B,T,H)
        out = self.drop(out)
        logits = self.fc(out)    # (B,T,V)
        return logits

## Sampling utilities

In [5]:
@torch.no_grad()
def sample_ids(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 120,
               temperature: float = 1.0, top_k: int = 50) -> List[int]:
    model.eval()
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]  # (1,V)
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            probs = torch.softmax(v, dim=-1)
            next_local = torch.multinomial(probs, num_samples=1)
            next_id = ix.gather(-1, next_local)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        next_id_int = int(next_id.item())
        x = torch.cat([x, torch.tensor([[next_id_int]], device=DEVICE)], dim=1)

        if eos_id is not None and next_id_int == eos_id:
            break
        if x.size(1) > 512:
            x = x[:, -512:]

    return x[0].tolist()

def decode(ids: List[int]) -> str:
    toks = [vocab[i] if 0 <= i < len(vocab) else '<unk>' for i in ids]
    text = ' '.join(toks)
    text = (text
            .replace(' ,', ',').replace(' .', '.').replace(' !', '!').replace(' ?', '?')
            .replace(' ;', ';').replace(' :', ':')
            .replace(' )', ')').replace('( ', '(')
            .replace(' - ', '-')
           )
    return text

## Train + evaluate

In [6]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

def run_eval(model, loader=None) -> dict:
    """Evaluate model, return dict with loss, ppl, top1_acc, top5_acc, mrr."""
    if loader is None:
        loader = val_loader
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    total_top1 = 0
    total_top5 = 0
    total_rr = 0.0  # sum of reciprocal ranks

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast("cuda", enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok

            # Flatten for metric computation
            flat_logits = logits.reshape(-1, logits.size(-1))  # (B*T, V)
            flat_y = y.reshape(-1)                              # (B*T,)
            flat_mask = mask.reshape(-1)                        # (B*T,)

            # Top-1 accuracy
            pred_top1 = flat_logits.argmax(dim=-1)
            total_top1 += int(((pred_top1 == flat_y) & flat_mask).sum().item())

            # Top-5 accuracy
            _, pred_top5 = flat_logits.topk(5, dim=-1)  # (B*T, 5)
            target_exp = flat_y.unsqueeze(-1).expand_as(pred_top5)
            hits5 = (pred_top5 == target_exp).any(dim=-1) & flat_mask
            total_top5 += int(hits5.sum().item())

            # MRR (mean reciprocal rank) — rank = 1 + count of tokens scoring higher
            target_logits = flat_logits.gather(1, flat_y.unsqueeze(-1))  # (B*T, 1)
            ranks = (flat_logits > target_logits).sum(dim=-1).float()    # 0-indexed
            rr = 1.0 / (ranks + 1.0)
            total_rr += float((rr * flat_mask.float()).sum().item())

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    top1_acc = total_top1 / max(1, total_tokens)
    top5_acc = total_top5 / max(1, total_tokens)
    mrr = total_rr / max(1, total_tokens)
    return {
        'loss': avg_loss,
        'ppl': ppl,
        'top1_acc': top1_acc,
        'top5_acc': top5_acc,
        'mrr': mrr,
    }

## Checkpoint helpers

In [7]:
def make_run_dir(base_dir: str) -> str:
    """Create a new run subfolder named YYYYMMDDHHMMSS."""
    name = datetime.now().strftime('%Y%m%d%H%M%S')
    path = os.path.join(base_dir, name)
    os.makedirs(path, exist_ok=True)
    return path

def build_hyperparams_dict(cfg: dict) -> dict:
    """Collect all hyperparameters needed to reproduce a run."""
    return {
        'seed': SEED,
        'seq_len': SEARCH_SEQ_LEN,
        'stride': SEARCH_STRIDE,
        'batch_size': SEARCH_BATCH_SIZE,
        'grad_clip': GRAD_CLIP,
        'emb_dim': SEARCH_EMB_DIM,
        'num_layers': cfg['num_layers'],
        'vocab_size': len(vocab),
        'lr': cfg['lr'],
        'hidden_size': cfg['hidden_size'],
        'dropout': cfg['dropout'],
        'steps_per_epoch': SEARCH_STEPS_PER_EPOCH,
        'max_epochs': SEARCH_EPOCHS,
        'early_stop_patience': EARLY_STOP_PATIENCE,
        'sched_factor': SCHED_FACTOR,
        'sched_patience': SCHED_PATIENCE,
        'sched_min_lr': SCHED_MIN_LR,
        'train_ratio': TRAIN_RATIO,
        'val_ratio': VAL_RATIO,
        'test_ratio': TEST_RATIO,
    }

def save_checkpoint(
    run_dir: str,
    filename: str,
    model: nn.Module,
    optimizer,
    scaler,
    scheduler,
    epoch: int,
    global_step: int,
    train_loss: float,
    val_loss: Optional[float],
    val_ppl: Optional[float],
    hyperparams: dict,
    is_step_ckpt: bool,
):
    """Save a checkpoint to run_dir/filename."""
    path = os.path.join(run_dir, filename)
    ckpt = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'epoch': epoch,
        'global_step': global_step,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_ppl': val_ppl,
        'hyperparams': hyperparams,
        'is_step_ckpt': is_step_ckpt,
    }
    torch.save(ckpt, path)
    return path

def prune_checkpoints(run_dir: str, threshold: float):
    """Delete epoch checkpoints whose val_loss exceeds best_val_loss + threshold.
    Step checkpoints are always deleted (they are crash recovery only)."""
    ckpt_files = sorted(glob.glob(os.path.join(run_dir, '*.pt')))
    if not ckpt_files:
        return

    epoch_ckpts = []  # (path, val_loss)
    for f in ckpt_files:
        ckpt = torch.load(f, map_location='cpu', weights_only=False)
        if ckpt.get('is_step_ckpt', False):
            os.remove(f)
            print(f"    pruned step ckpt: {os.path.basename(f)}")
        else:
            val_loss = ckpt.get('val_loss')
            if val_loss is not None:
                epoch_ckpts.append((f, val_loss))

    if not epoch_ckpts:
        return

    best_val = min(vl for _, vl in epoch_ckpts)
    for f, vl in epoch_ckpts:
        if vl > best_val + threshold:
            os.remove(f)
            print(f"    pruned epoch ckpt: {os.path.basename(f)} (val_loss={vl:.4f}, best={best_val:.4f}, threshold={threshold})")

print("Checkpoint helpers ready.")

Checkpoint helpers ready.


## Tests

In [8]:
def encode_prompt(text: str) -> List[int]:
    toks = text.split()
    ids = [tok2id.get(t, tok2id.get('<unk>', 0)) for t in toks]
    return ids

In [ ]:
import itertools

# ---------- Search space ----------
SEARCH_GRID = {
    'lr':          [0.02],
    'hidden_size': [512],
    'dropout':     [0.2],
    'num_layers':  [50],
}

SEARCH_EMB_DIM    = 256
SEARCH_EPOCHS     = 50
SEARCH_STEPS_PER_EPOCH = 1500
SEARCH_BATCH_SIZE = 256
SEARCH_SEQ_LEN    = 50
SEARCH_STRIDE     = 10
EARLY_STOP_PATIENCE = 4

# ReduceLROnPlateau settings
SCHED_FACTOR  = 0.5            # halve LR when plateau
SCHED_PATIENCE = 1             # reduce after 1 epoch of no improvement
SCHED_MIN_LR  = 1e-5           # don't go below this

# Build all configs
keys = list(SEARCH_GRID.keys())
combos = list(itertools.product(*[SEARCH_GRID[k] for k in keys]))
configs = [dict(zip(keys, c)) for c in combos]

print(f"Total configurations: {len(configs)}")
for i, cfg in enumerate(configs):
    print(f"  [{i+1}] lr={cfg['lr']}, hidden={cfg['hidden_size']}, dropout={cfg['dropout']}, layers={cfg['num_layers']}")

Total configurations: 1
  [1] lr=0.02, hidden=512, dropout=0.2, layers=50


: 

In [ ]:
def run_search_eval(mdl, loader=None) -> dict:
    """Evaluate model on a given loader, return dict with loss, ppl, top1_acc, top5_acc, mrr."""
    if loader is None:
        loader = val_loader
    mdl.eval()
    total_loss = 0.0
    total_tokens = 0
    total_top1 = 0
    total_top5 = 0
    total_rr = 0.0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok

            flat_logits = logits.reshape(-1, logits.size(-1))
            flat_y = y.reshape(-1)
            flat_mask = mask.reshape(-1)

            # Top-1 accuracy
            pred_top1 = flat_logits.argmax(dim=-1)
            total_top1 += int(((pred_top1 == flat_y) & flat_mask).sum().item())

            # Top-5 accuracy
            _, pred_top5 = flat_logits.topk(5, dim=-1)
            target_exp = flat_y.unsqueeze(-1).expand_as(pred_top5)
            hits5 = (pred_top5 == target_exp).any(dim=-1) & flat_mask
            total_top5 += int(hits5.sum().item())

            # MRR (mean reciprocal rank) — rank = 1 + count of tokens scoring higher
            target_logits = flat_logits.gather(1, flat_y.unsqueeze(-1))  # (B*T, 1)
            ranks = (flat_logits > target_logits).sum(dim=-1).float()    # 0-indexed
            rr = 1.0 / (ranks + 1.0)
            total_rr += float((rr * flat_mask.float()).sum().item())

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    top1_acc = total_top1 / max(1, total_tokens)
    top5_acc = total_top5 / max(1, total_tokens)
    mrr = total_rr / max(1, total_tokens)
    return {'loss': avg_loss, 'ppl': ppl, 'top1_acc': top1_acc, 'top5_acc': top5_acc, 'mrr': mrr}


# ---------- Resume handling ----------
resume_ckpt = None
if RESUME_FROM is not None:
    if not os.path.exists(RESUME_FROM):
        raise FileNotFoundError(f"Resume checkpoint not found: {RESUME_FROM}")
    resume_ckpt = torch.load(RESUME_FROM, map_location='cpu', weights_only=False)
    resume_run_dir = os.path.dirname(RESUME_FROM)
    rh = resume_ckpt['hyperparams']
    configs = [{
        'lr': rh['lr'],
        'hidden_size': rh['hidden_size'],
        'dropout': rh['dropout'],
        'num_layers': rh['num_layers'],
    }]
    SEARCH_EMB_DIM = rh['emb_dim']
    print(f"Resuming from: {RESUME_FROM}")
    print(f"  epoch={resume_ckpt['epoch']}, global_step={resume_ckpt['global_step']}, "
          f"train_loss={resume_ckpt['train_loss']:.4f}")
    if resume_ckpt.get('val_loss') is not None:
        print(f"  val_loss={resume_ckpt['val_loss']:.4f}, val_ppl={resume_ckpt['val_ppl']:.2f}")

search_results = []
best_model_state = None

actual_steps = min(SEARCH_STEPS_PER_EPOCH, len(train_loader))

for cfg_idx, cfg in enumerate(configs):
    print(f"\n{'='*70}")
    print(f"Config [{cfg_idx+1}/{len(configs)}]  lr={cfg['lr']}  hidden={cfg['hidden_size']}  "
          f"dropout={cfg['dropout']}  layers={cfg['num_layers']}")
    print(f"Steps per epoch: {actual_steps} (loader has {len(train_loader)} batches, cap {SEARCH_STEPS_PER_EPOCH})")
    print('='*70)

    set_seed(SEED)
    hparams = build_hyperparams_dict(cfg)

    # Create or reuse run directory
    if resume_ckpt is not None and cfg_idx == 0:
        run_dir = resume_run_dir
    else:
        run_dir = make_run_dir(CKPT_DIR)
    print(f"Run dir: {run_dir}")

    with open(os.path.join(run_dir, 'hyperparams.json'), 'w') as f:
        json.dump(hparams, f, indent=2)

    mdl = LSTMLM(
        vocab_size=len(vocab),
        emb_dim=SEARCH_EMB_DIM,
        hidden=cfg['hidden_size'],
        num_layers=cfg['num_layers'],
        dropout=cfg['dropout'],
        pad_id=pad_id,
    ).to(DEVICE)

    n_params = sum(p.numel() for p in mdl.parameters())
    print(f"Model params: {n_params:,}")

    crit = nn.CrossEntropyLoss(ignore_index=pad_id)
    opt  = torch.optim.AdamW(mdl.parameters(), lr=cfg['lr'], weight_decay=0.01)
    scl  = GradScaler("cuda", enabled=use_amp)

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode='min', factor=SCHED_FACTOR,
        patience=SCHED_PATIENCE, min_lr=SCHED_MIN_LR
    )

    start_epoch = 1
    global_step = 0
    best_val_loss = float('inf')
    patience_counter = 0
    best_epoch = 0
    best_state = None
    best_metrics = None

    if resume_ckpt is not None and cfg_idx == 0:
        mdl.load_state_dict(resume_ckpt['model_state_dict'])
        opt.load_state_dict(resume_ckpt['optimizer_state_dict'])
        scl.load_state_dict(resume_ckpt['scaler_state_dict'])
        scheduler.load_state_dict(resume_ckpt['scheduler_state_dict'])
        global_step = resume_ckpt['global_step']

        if resume_ckpt.get('is_step_ckpt', False):
            start_epoch = resume_ckpt['epoch']
            print(f"  Resuming mid-epoch: restarting epoch {start_epoch} from step 0")
        else:
            start_epoch = resume_ckpt['epoch'] + 1
            print(f"  Resuming after epoch {resume_ckpt['epoch']}: starting epoch {start_epoch}")

        for f in sorted(glob.glob(os.path.join(run_dir, 'epoch_*.pt'))):
            c = torch.load(f, map_location='cpu', weights_only=False)
            vl = c.get('val_loss')
            if vl is not None and vl < best_val_loss:
                best_val_loss = vl
                best_epoch = c['epoch']
                best_state = c['model_state_dict']
        if best_val_loss < float('inf'):
            print(f"  Restored best_val_loss={best_val_loss:.4f} from epoch {best_epoch}")

        resume_ckpt = None

    for epoch in range(start_epoch, SEARCH_EPOCHS + 1):
        mdl.train()
        running = 0.0
        seen_tokens = 0

        pbar = tqdm(enumerate(train_loader, start=1),
                     total=actual_steps,
                     desc=f"  E{epoch}/{SEARCH_EPOCHS}", leave=False)

        for step, (x, y) in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = crit(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            scl.scale(loss).backward()
            scl.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), GRAD_CLIP)
            scl.step(opt)
            scl.update()

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            running += float(loss.item()) * tok
            seen_tokens += tok
            global_step += 1

            avg = running / max(1, seen_tokens)
            pbar.set_postfix(loss=f"{avg:.4f}", step=global_step)

            if SAVE_EVERY_STEPS > 0 and step % SAVE_EVERY_STEPS == 0:
                step_fname = f"step_{global_step:07d}.pt"
                save_checkpoint(
                    run_dir, step_fname, mdl, opt, scl, scheduler,
                    epoch=epoch, global_step=global_step,
                    train_loss=avg, val_loss=None, val_ppl=None,
                    hyperparams=hparams, is_step_ckpt=True,
                )

            if step >= actual_steps:
                break
        pbar.close()

        # --- Epoch-end: evaluate with all metrics ---
        val_m = run_search_eval(mdl)
        train_loss = running / max(1, seen_tokens)
        current_lr = opt.param_groups[0]['lr']
        print(f"  epoch {epoch}  train_loss={train_loss:.4f}  "
              f"val_loss={val_m['loss']:.4f}  val_ppl={val_m['ppl']:.2f}  "
              f"top1={val_m['top1_acc']:.4f}  top5={val_m['top5_acc']:.4f}  "
              f"mrr={val_m['mrr']:.4f}  lr={current_lr:.6f}", end="")

        epoch_fname = f"epoch_{epoch:03d}.pt"
        save_checkpoint(
            run_dir, epoch_fname, mdl, opt, scl, scheduler,
            epoch=epoch, global_step=global_step,
            train_loss=train_loss, val_loss=val_m['loss'], val_ppl=val_m['ppl'],
            hyperparams=hparams, is_step_ckpt=False,
        )

        scheduler.step(val_m['loss'])

        if val_m['loss'] < best_val_loss:
            best_val_loss = val_m['loss']
            best_epoch = epoch
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in mdl.state_dict().items()}
            best_metrics = val_m.copy()
            print(f"  *best* [saved {epoch_fname}]")
        else:
            patience_counter += 1
            print(f"  (no improve {patience_counter}/{EARLY_STOP_PATIENCE}) [saved {epoch_fname}]")
            if patience_counter >= EARLY_STOP_PATIENCE:
                print(f"  >> early stop at epoch {epoch}, best was epoch {best_epoch}")
                break

        print(f"  Pruning checkpoints (threshold={PRUNE_VAL_LOSS_THRESHOLD})...")
        prune_checkpoints(run_dir, PRUNE_VAL_LOSS_THRESHOLD)

    search_results.append({
        **cfg,
        'best_val_loss': best_val_loss,
        'best_val_ppl': math.exp(min(20.0, best_val_loss)),
        'best_top1': best_metrics['top1_acc'] if best_metrics else 0.0,
        'best_top5': best_metrics['top5_acc'] if best_metrics else 0.0,
        'best_mrr': best_metrics['mrr'] if best_metrics else 0.0,
        'best_epoch': best_epoch,
        'total_epochs': epoch,
        'best_state': best_state,
        'run_dir': run_dir,
    })

    del mdl, opt, scl, crit, scheduler
    torch.cuda.empty_cache()

print("\n\nGrid search complete.")


Config [1/1]  lr=0.02  hidden=512  dropout=0.2  layers=50
Steps per epoch: 1011 (loader has 1011 batches, cap 1500)
Run dir: checkpoints\20260314200537
Model params: 124,752,046


  E1/50:   0%|          | 0/1011 [00:00<?, ?it/s]

  epoch 1  train_loss=7.0120  val_loss=7.3173  val_ppl=1506.11  top1=0.0663  top5=0.2198  mrr=0.1438  lr=0.020000  *best* [saved epoch_001.pt]
  Pruning checkpoints (threshold=0.1)...
    pruned step ckpt: step_0000100.pt
    pruned step ckpt: step_0000200.pt
    pruned step ckpt: step_0000300.pt
    pruned step ckpt: step_0000400.pt
    pruned step ckpt: step_0000500.pt
    pruned step ckpt: step_0000600.pt
    pruned step ckpt: step_0000700.pt
    pruned step ckpt: step_0000800.pt
    pruned step ckpt: step_0000900.pt
    pruned step ckpt: step_0001000.pt


  E2/50:   0%|          | 0/1011 [00:00<?, ?it/s]

  epoch 2  train_loss=6.9910  val_loss=7.3388  val_ppl=1538.86  top1=0.0575  top5=0.2031  mrr=0.1374  lr=0.020000  (no improve 1/4) [saved epoch_002.pt]
  Pruning checkpoints (threshold=0.1)...
    pruned step ckpt: step_0001111.pt
    pruned step ckpt: step_0001211.pt
    pruned step ckpt: step_0001311.pt
    pruned step ckpt: step_0001411.pt
    pruned step ckpt: step_0001511.pt
    pruned step ckpt: step_0001611.pt
    pruned step ckpt: step_0001711.pt
    pruned step ckpt: step_0001811.pt
    pruned step ckpt: step_0001911.pt
    pruned step ckpt: step_0002011.pt


...

  epoch 21  train_loss=6.3454  val_loss=6.5126  val_ppl=673.55  top1=0.0663  top5=0.2199  mrr=0.1448  lr=0.005000  *best* [saved epoch_021.pt]
  Pruning checkpoints (threshold=0.1)...
    pruned step ckpt: step_0020320.pt
    pruned step ckpt: step_0020420.pt
    pruned step ckpt: step_0020520.pt
    pruned step ckpt: step_0020620.pt
    pruned step ckpt: step_0020720.pt
    pruned step ckpt: step_0020820.pt
    pruned step ckpt: step_0020920.pt
    pruned step ckpt: step_0021020.pt
    pruned step ckpt: step_0021120.pt
    pruned step ckpt: step_0021220.pt


  E22/50:   0%|          | 0/1011 [00:00<?, ?it/s]

  epoch 22  train_loss=6.3447  val_loss=6.5132  val_ppl=673.96  top1=0.0663  top5=0.2208  mrr=0.1480  lr=0.005000  (no improve 1/4) [saved epoch_022.pt]
  Pruning checkpoints (threshold=0.1)...
    pruned step ckpt: step_0021331.pt
    pruned step ckpt: step_0021431.pt
    pruned step ckpt: step_0021531.pt
    pruned step ckpt: step_0021631.pt
    pruned step ckpt: step_0021731.pt
    pruned step ckpt: step_0021831.pt
    pruned step ckpt: step_0021931.pt
    pruned step ckpt: step_0022031.pt
    pruned step ckpt: step_0022131.pt
    pruned step ckpt: step_0022231.pt


  E23/50:   0%|          | 0/1011 [00:00<?, ?it/s]

  epoch 23  train_loss=6.3425  val_loss=6.5113  val_ppl=672.67  top1=0.0663  top5=0.2198  mrr=0.1452  lr=0.005000  *best* [saved epoch_023.pt]
  Pruning checkpoints (threshold=0.1)...


In [ ]:
# Sort by best validation loss and display results
search_results.sort(key=lambda r: r['best_val_loss'])

print(f"{'Rank':<5} {'LR':<10} {'Hidden':<8} {'Layers':<8} {'Drop':<7} {'Val Loss':<10} {'Val PPL':<10} "
      f"{'Top1':<8} {'Top5':<8} {'MRR':<8} {'Best@':<7} {'Stopped':<7} {'Run Dir'}")
print('-' * 120)
for i, r in enumerate(search_results):
    print(f"{i+1:<5} {r['lr']:<10} {r['hidden_size']:<8} {r['num_layers']:<8} {r['dropout']:<7.2f} "
          f"{r['best_val_loss']:<10.4f} {r['best_val_ppl']:<10.2f} "
          f"{r['best_top1']:<8.4f} {r['best_top5']:<8.4f} {r['best_mrr']:<8.4f} "
          f"{r['best_epoch']:<7} {r['total_epochs']:<7} {r['run_dir']}")

best = search_results[0]
print(f"\nBest config:")
print(f"  lr={best['lr']}, hidden_size={best['hidden_size']}, num_layers={best['num_layers']}, dropout={best['dropout']}")
print(f"  val_loss={best['best_val_loss']:.4f}, val_ppl={best['best_val_ppl']:.2f}, "
      f"top1={best['best_top1']:.4f}, top5={best['best_top5']:.4f}, mrr={best['best_mrr']:.4f}")
print(f"  best at epoch {best['best_epoch']}, run_dir={best['run_dir']}")

# --- Final test evaluation (only done once, on best model) ---
print(f"\n{'='*60}")
print("Final TEST evaluation (best model from grid search)")
print('='*60)

best_mdl = LSTMLM(
    vocab_size=len(vocab),
    emb_dim=SEARCH_EMB_DIM,
    hidden=best['hidden_size'],
    num_layers=best['num_layers'],
    dropout=best['dropout'],
    pad_id=pad_id,
).to(DEVICE)
best_mdl.load_state_dict(best['best_state'])

test_m = run_search_eval(best_mdl, loader=test_loader)
print(f"  test_loss={test_m['loss']:.4f}  test_ppl={test_m['ppl']:.2f}  "
      f"top1={test_m['top1_acc']:.4f}  top5={test_m['top5_acc']:.4f}  mrr={test_m['mrr']:.4f}")

# List remaining checkpoints for best run
print(f"\nCheckpoints in {best['run_dir']}:")
for f in sorted(glob.glob(os.path.join(best['run_dir'], '*.pt'))):
    print(f"  {os.path.basename(f)}")

# Clean up saved states from search_results to free memory
for r in search_results:
    r.pop('best_state', None)
del best_mdl
torch.cuda.empty_cache()